<a href="https://colab.research.google.com/github/NarmataGoel/NarmataGoel/blob/main/UCS420_LA9_Starterassigin7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [24]:
ROLL_NO = 1024170023         # <-- your full roll number
NAME = "Narmata Goel"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [25]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [26]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [27]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170023   Name: Narmata Goel   Fingerprint: C9B3C22D   Tickets: 15

Ticket_ID                                                                                 Ticket_Text
      T01                                   The student portal is running slowly. Can you check this?
      T02                                                     The VPN is not working. This is urgent.
      T03                   Laptop stopped responding before the assignment deadline. This is urgent.
      T04                  Password was working yesterday during the lab. I have tried several times!
      T05     I was locked out of the VPN after resetting my password. The same problem occurs again.
      T06                                   The projector was working yesterday. It worked yesterday.
      T07                                       The printer keeps disconnecting. It worked yesterday.
      T08                                                             Is it fixed?? Pls reply ASAP :

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [28]:
# Q1: Preprocessing

rows = []

# Run process_ticket() on every ticket
for _, row in df.iterrows():
    result = process_ticket(row["Ticket_Text"])

    rows.append({
        "Ticket_ID": row["Ticket_ID"],
        "Number of Sentences": len(result["sentences"]),
        "Number of Tokens": len(result["tokens"]),
        "Number of Punctuation Tokens": len(result["punct"]),
        "Number of Stopwords": len(result["stops"]),
        "Number of Unique Tokens": len(set(result["tokens"])),
        "Number of Final Tokens": len(result["final"])
    })

# Create table
q1_table = pd.DataFrame(rows)

# Add TOTAL row
total_row = {
    "Ticket_ID": "TOTAL",
    "Number of Sentences": q1_table["Number of Sentences"].sum(),
    "Number of Tokens": q1_table["Number of Tokens"].sum(),
    "Number of Punctuation Tokens": q1_table["Number of Punctuation Tokens"].sum(),
    "Number of Stopwords": q1_table["Number of Stopwords"].sum(),
    "Number of Unique Tokens": q1_table["Number of Unique Tokens"].sum(),
    "Number of Final Tokens": q1_table["Number of Final Tokens"].sum()
}

q1_table = pd.concat(
    [q1_table, pd.DataFrame([total_row])],
    ignore_index=True
)

print("Q1 - Preprocessing Table")
display(q1_table)

# Select one focus ticket
selected_id = FOCUS[0]

selected_text = df.loc[
    df["Ticket_ID"] == selected_id,
    "Ticket_Text"
].iloc[0]

result = process_ticket(selected_text)

print("\nSelected Ticket:", selected_id)
print("Ticket Text:", seected_text)

print("\nActual Tokens:")
print(result["tokens"])

print("\nPunctuation Tokens:")
print(result["punct"])

print("\nStopwords:")
print(result["stops"])

print("\nFinal Tokens:")
print(result["final"])

Q1 - Preprocessing Table


,Ticket_ID,Number of Sentences,Number of Tokens,Number of Punctuation Tokens,Number of Stopwords,Number of Unique Tokens,Number of Final Tokens
0,T01,2,12,2,5,12,5
1,T02,2,10,2,5,8,3
2,T03,2,12,2,4,11,6
3,T04,2,14,2,5,14,7
4,T05,2,18,2,10,16,6
5,T06,2,10,2,3,8,5
6,T07,2,9,2,2,8,5
7,T08,2,10,4,2,9,4
8,T09,1,7,2,1,7,4
9,T10,2,14,2,7,13,5



Selected Ticket: T01
Ticket Text: The student portal is running slowly. Can you check this?

Actual Tokens:
['the', 'student', 'portal', 'is', 'running', 'slowly', '.', 'can', 'you', 'check', 'this', '?']

Punctuation Tokens:
['.', '?']

Stopwords:
['the', 'is', 'can', 'you', 'this']

Final Tokens:
['student', 'portal', 'running', 'slowly', 'check']


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [29]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Q2: Regular Expressions

import re

# ---------------------------------------------------
# 1. Regex patterns
# ---------------------------------------------------

# Words longer than 5 letters
long_words_pattern = r"\b[A-Za-z]{6,}\b"

# Numbers: integers, decimals, versions, comma-separated numbers
numbers_pattern = r"\b(?:\d{1,3}(?:,\d{3})+|\d+(?:\.\d+)+|\d+(?:\.\d+)?)\b"

# Capitalized words
capitalized_pattern = r"\b[A-Z][A-Za-z]*\b"

# Words beginning with a vowel
vowel_words_pattern = r"\b[AEIOUaeiou][A-Za-z]*\b"


# ---------------------------------------------------
# 2. Replacement patterns
# ---------------------------------------------------

# Email
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

# URL
url_pattern = r"https?://[^\s]+|www\.[^\s]+"

# Phone numbers
phone_pattern = r"(?:\+\d{1,3}\s?)?\d{5}[\s-]\d{5}|\b\d{4}-\d{3}-\d{4}\b"


# ---------------------------------------------------
# 3. Test on TEST_LINES
# ---------------------------------------------------

for i, text in enumerate(TEST_LINES, start=1):

    print("=" * 70)
    print("TEST LINE", i)
    print("=" * 70)

    print("\nOriginal:")
    print(text)

    print("\nWords longer than 5 letters:")
    print(re.findall(long_words_pattern, text))

    print("\nNumbers:")
    print(re.findall(numbers_pattern, text))

    print("\nCapitalized words:")
    print(re.findall(capitalized_pattern, text))

    print("\nWords beginning with a vowel:")
    print(re.findall(vowel_words_pattern, text))

    print("\nAfter replacing EMAIL:")
    print(re.sub(email_pattern, "<EMAIL>", text))

    print("\nAfter replacing URL:")
    print(re.sub(url_pattern, "<URL>", text))

    print("\nAfter replacing PHONE:")
    print(re.sub(phone_pattern, "<PHONE>", text))

TEST LINE 1

Original:
Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.

Words longer than 5 letters:
['helpdesk', 'thapar', 'thapar', 'thapar']

Numbers:
[]

Capitalized words:
['Mail']

Words beginning with a vowel:
['it', 'edu', 'or', 'a', 'univ', 'ac', 'in', 'edu', 'or', 'edu', 'it']

After replacing EMAIL:
Mail <EMAIL> or <EMAIL>; visit https://lms.thapar.edu/help or www.thapar.edu/it.

After replacing URL:
Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit <URL> or <URL>

After replacing PHONE:
Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
TEST LINE 2

Original:
Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.

Words longer than 5 letters:
['Version']

Numbers:
['91', '98765', '43210', '91', '98765', '43210', '0175', '239', '3201', '2.0.1', '95.5', '1,250']

Capitalized words:
['Call', 'Version', 'R

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:*

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [30]:
# Q3: Custom tokenizer

from nltk.tokenize import RegexpTokenizer, word_tokenize

# One RegexpTokenizer pattern
pattern = r"""
https?://[^\s]+|www\.[^\s]+|                         # URLs
[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|     # Emails
(?:\+\d{1,3}[-\s]?)?\d{5}[-\s]\d{5}|                # Phone numbers
\d+(?:\.\d+)+|                                      # Versions/decimals
[A-Za-z]+(?:-[A-Za-z]+)+|                            # Hyphenated words
[A-Za-z]+(?:'[A-Za-z]+)+|                            # Contractions
[A-Za-z]+|                                           # Normal words
[^\w\s]                                              # Punctuation
"""

tokenizer = RegexpTokenizer(pattern, flags=re.VERBOSE)


def my_tokenize(text):
    return tokenizer.tokenize(text)


# ---------------------------------------------------
# Test on TEST_LINES
# ---------------------------------------------------

for i, text in enumerate(TEST_LINES, start=1):
    print("=" * 60)
    print("TEST LINE", i)
    print("=" * 60)

    print("\nMy tokenizer:")
    print(my_tokenize(text))

    print("\nword_tokenize:")
    print(word_tokenize(text))

TEST LINE 1

My tokenizer:
['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', ';', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']

word_tokenize:
['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']
TEST LINE 2

My tokenizer:
['Call', '+91 98765 43210', ',', '+91-98765-43210', 'or', '-', '-', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', ',', '.']

word_tokenize:
['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']
TEST LINE 3

My tokenizer:
['The', 'state-of-the-art', 'lab', "isn't", 'open', ';', 'log-in', 'at', '5.30', 'p', '.', 'm', '.', "didn't", 'work', '.']

word_tokenize:
['The', 'state-of-the-art', 'lab', 'is', "n't", 'open', ';', 'log-in', 'at', '5.30', 'p.m.', 'did', "n't", 'work', '.']


**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Your code

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Your code

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:*